# 03. Q-routing과 변화 후 재학습
목표: 논문이 비교하는 Q-routing의 직관을 작은 비용 모델에서 확인한다. Python 3.10+ 표준 라이브러리, 위에서부터 실행. [논문](https://doi.org/10.5281/zenodo.13885645).

**중요:** 이 notebook은 목적지 D 하나를 가진 표 기반 toy Q-routing이다. DDQN, 심층신경망, 위성 궤도, 실제 큐 관측, SFL은 구현하지 않는다. 고정 양의 비용(ms)을 이용하며 노드는 이웃 비용 추정치를 받아 갱신한다. 중앙의 Python 객체가 실습을 편하게 구성하지만 정책은 현재 노드의 이웃 행동만 선택한다.

학습용 갱신: `Q_i(j) += alpha * (cost(i,j) + min_k Q_j(k) - Q_i(j))`. 목적지에서는 다음 값 0, gamma=1. 비용 최소화이므로 min이며, 논문 본문에 제시된 원식이라고 주장하지 않는다. 양의 간선 비용과 terminal 상태를 이용하고 max_hops로 루프를 제한한다.

In [ ]:
import random, copy, heapq, math
graph = {'A': {'B': 2.0, 'C': 6.0, 'D': 50.0},
         'B': {'A': 2.0, 'D': 2.0}, 'C': {'A': 6.0, 'D': 6.0},
         'D': {'A': 50.0, 'B': 2.0, 'C': 6.0}}
def train(g, q=None, episodes=5000, seed=7, alpha=.2, epsilon=.2):
    if not 0 < alpha <= 1 or not 0 <= epsilon <= 1:
        raise ValueError('invalid alpha or epsilon')
    if any(not math.isfinite(c) or c <= 0 for links in g.values() for c in links.values()):
        raise ValueError('positive finite costs required')
    rng = random.Random(seed)
    q = copy.deepcopy(q) if q is not None else {}
    for node in g:
        q.setdefault(node, {})
        for nxt in g[node]:
            q[node].setdefault(nxt, 0.0)  # 미방문 비용은 optimistic zero
    capped = 0
    for _ in range(episodes):
        node = 'A'
        for hop in range(32):
            actions = sorted(g[node])  # 현재 연결만 사용: stale action을 mask
            if not actions:
                capped += 1
                break
            nxt = rng.choice(actions) if rng.random() < epsilon else min(actions, key=lambda a:(q[node][a], a))
            dead_end = nxt != 'D' and not g[nxt]
            future = 0.0 if nxt == 'D' else (1000.0 if dead_end else min(q[nxt][a] for a in g[nxt]))
            q[node][nxt] += alpha * (g[node][nxt] + future - q[node][nxt])
            node = nxt
            if dead_end:
                capped += 1  # 연결 단절에는 교육용 1000ms 벌점을 적용
                break
            if node == 'D':
                break
        else:
            capped += 1
    return q, capped
def evaluate(g, q):
    # 탐험과 값 갱신 없이 평가하여 학습과 평가를 분리한다.
    node, path, cost = 'A', ['A'], 0.0
    for _ in range(32):
        if node == 'D':
            return {'delivered': True, 'cost_ms': cost, 'path': path}
        if not g[node]:
            break
        nxt = min(g[node], key=lambda a:(q[node].get(a, 0.0), a))
        cost += g[node][nxt]
        path.append(nxt)
        node = nxt
    return {'delivered': node == 'D', 'cost_ms': cost, 'path': path}
def oracle(g):
    heap, best = [(0.0, 'A')], {'A': 0.0}
    while heap:
        cost, node = heapq.heappop(heap)
        if cost > best[node]:
            continue
        if node == 'D':
            return cost
        for nxt, edge in g[node].items():
            if cost + edge < best.get(nxt, math.inf):
                best[nxt] = cost + edge
                heapq.heappush(heap, (cost + edge, nxt))
    return math.inf


In [ ]:
q, capped = train(graph)
saved = copy.deepcopy(q)
before = evaluate(graph, q)
assert q == saved  # 평가 과정에서 Q-table이 바뀌지 않아야 한다.
assert before['delivered'] and before['cost_ms'] == oracle(graph) == 4.0
print('before change:', before, 'training capped=', capped)
changed = copy.deepcopy(graph)
del changed['B']['D']
del changed['D']['B']
frozen = evaluate(changed, q)
print('frozen policy after link removal:', frozen)
adapted, capped = train(changed, q=q, episodes=8000, seed=11)
after = evaluate(changed, adapted)
assert after['delivered'] and after['cost_ms'] == oracle(changed) == 12.0
assert after['path'] == ['A', 'C', 'D']
assert all(v in changed[u] for u,v in zip(after['path'], after['path'][1:]))
print('adapted:', after, 'training capped=', capped)
for seed in range(5):
    learned, _ = train(graph, seed=seed)
    report = evaluate(graph, learned)
    assert report['delivered'] and report['cost_ms'] == 4.0
print('5-seed baseline checks passed')
# max-hop 제한이 없으면 순환 정책이 검증 실행을 끝내지 못할 수 있다.
cyclic_q = {u:{v:0.0 for v in graph[u]} for u in graph}
loop = evaluate(graph, cyclic_q)
assert not loop['delivered'] and len(loop['path']) == 33
isolated = copy.deepcopy(graph)
isolated['A'] = {}
assert not evaluate(isolated, q)['delivered']
assert math.isinf(oracle(isolated))


## 해석과 연구 수준 확장
Dijkstra oracle은 전체 현재 비용을 안다. 이 작은 결정론적 예에서 학습 정책이 oracle과 같아졌다고 원 논문 성능을 재현한 것은 아니다. 링크 변화 직후 낡은 Q 값은 루프를 만들 수 있다. max_hops 초과는 전달 실패로 집계하고, 실패한 경로의 비용을 성공 E2E 평균에 넣지 않는다.

과제: (1) 학습하지 않은 링크를 새로 추가하고 epsilon의 효과를 비교, (2) 02의 실제 큐에서 관측한 one-hop 비용을 feedback으로 연결, (3) 관측 지연·메시지 비용 추가, (4) 독립 seed별 평균·p95·전달률·제어 overhead 함께 보고.

DDQN으로 넘어갈 때는 상태·action mask·보상 부호·replay sampling·target update·terminal bootstrap을 명시해야 한다. 이 구현을 그대로 DNN이라고 이름만 바꾸지 않는다. SFL 및 CKA도 별도 검증 과제이며 여기에서 실행하지 않았다.